# Практика · Тема 02 · Токенізація

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє завдання: [homework.html](homework.html)

Тут ми надрукуємо **кожне число, яке стоїть у лекції**. Нічого не береться зі стелі:
всі таблиці лекції — це вивід клітинок нижче.

Що зробимо:

1. Зберемо **паралельний корпус**: англійський оригінал і український переклад того
   самого рядка лежать поруч.
2. Порівняємо **три рівні різання** — символи, слова, субслова — за розміром словника
   й за часткою невідомих слів.
3. Напишемо **BPE своїми руками** й звіримо результат із бібліотекою `tokenizers`.
4. Заміряємо, **у скільки разів дорожча українська** й **як ця націнка залежить від
   розміру словника**.
5. Розділимо націнку на два внески: довжина слів і дрібність різання.
6. Подивимось, що токенізатор знає **домен**, а не мову.
7. Порівняємо **WordPiece з BPE**, додамо **спецтокени** й порахуємо **ціну довгого
   тексту**.

> ⏱ Зошит навчає близько шістдесяти токенізаторів. Заміряно: **близько трьох хвилин**
> на чотирьох ядрах без відеокарти. Найдовше йде розділ 5 — там шість розмірів
> словника на двох мовах і трьох зернах.

> 🎲 Скрізь, де є випадковість (а вона тут одна — як саме корпус ділиться на
> навчальну й тестову частини), ми беремо **три зерна** й показуємо середнє
> та розкид. Різниця, менша за розкид, різницею не є.

In [ ]:
import sys
import glob
import gettext
import random
import collections
import statistics
import time

import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors, decoders

# фіксуємо версії: числа лекції отримані саме на них
print("Python     ", sys.version.split()[0])
print("tokenizers ", tokenizers.__version__)

STARTED_AT = time.time()

## 1 · Корпус: той самий зміст двома мовами

Головне число теми — **у скільки разів українська дорожча в токенах, ніж англійська**.
Щоб таке порівняння взагалі щось означало, обидві мови мусять говорити **те саме**.
Інакше ми порівняємо не мови, а тексти.

Такий корпус у системі вже лежить. Кожна програма з графічним інтерфейсом носить
із собою файли перекладу `.mo`, і в кожному записі поруч стоять **англійський
оригінал і український переклад** — рядок у рядок.

Це вузький домен: технічна лексика, короткі речення, наказовий спосіб. Робити з
нього висновки про мову взагалі не можна — і саме на цьому побудований розділ 7.

In [ ]:
LOCALE_GLOB = "/usr/share/locale/uk/LC_MESSAGES/*.mo"


def load_system_corpus():
    """Читає всі .mo-каталоги української локалі й повертає пари (оригінал, переклад).

    Беремо лише переклади довші за 30 символів: коротші — це підписи кнопок
    на одне слово, у яких немає ані речення, ані морфології.
    """
    pairs = []
    for path in sorted(glob.glob(LOCALE_GLOB)):
        try:
            with open(path, "rb") as handle:
                catalog = gettext.GNUTranslations(handle)
        except Exception:
            # зламаний або нечитаний файл не має валити весь зошит
            continue
        for source, target in catalog._catalog.items():
            if not isinstance(source, str) or not isinstance(target, str):
                continue
            if len(target) > 30 and "Project-Id" not in target:
                pairs.append((source, target))
    return pairs


system_pairs = load_system_corpus()
print("файлів локалі знайдено:", len(glob.glob(LOCALE_GLOB)))
print("пар «оригінал — переклад»:", len(system_pairs))

### Запобіжник: якщо української локалі в системі немає

На чужій машині цих файлів може не бути взагалі. Тоді зошит мусить не впасти, а
перейти на **вбудований мінікорпус** — 260 тих самих пар, збережених просто тут.

Мінікорпус у 350 разів менший за системний, тож числа на ньому будуть **іншими**,
ніж у лекції. Клітинка про це прямо скаже, а не промовчить.

In [ ]:
# запасний корпус: ті самі пари, але 260 замість 93 тисяч
MINI_CORPUS = [
    ('Stroke is averaged over selected objects', "Штрих усереднено для позначених об'єктів"),
    ('cannot specify -static with -fsanitize=thread', 'не можна вказувати -static з -fsanitize=thread'),
    ('string start/ends with forbidden hyphen', 'рядок починається або завершується на заборонений дефіс'),
    ('Authentication is required to change your own user data', 'Для зміни даних вашого користувача слід пройти розпізнавання'),
    ('list of group members to add', 'список учасників групи, яких слід додати'),
    ('only SSH or UNIX socket connection supported.', "передбачено підтримку лише з'єднань із SSH або сокетами UNIX."),
    ('Waiting for authentication', 'Очікування на завершення розпізнавання'),
    ('Ceará, Maranhão, Paraíba, Piauí, Rio Grande do Norte', 'Сеара, Мараньян, Параїба, Піауї, Ріо-Гранде-ду-Норті'),
    ('Expecting "tablespace OID/database OID/relation filenode".', 'Очікуємо "tablespace OID/database OID/relation filenode".'),
    ('transfer rate must be greater than zero', 'частота передач повинна бути більша за нуль'),
    ('window functions are not allowed in DEFAULT expressions', 'віконні функції не можна застосовувати у виразах DEFAULT'),
    ('show a list of all dependencies and what packages provide them', 'показати список усіх залежностей та пакунки, які їх надають'),
    ('North American Indian languages', 'Північноамериканські індіанські мови'),
    ('GSSAPI context establishment error', 'помилка встановлення контексту GSSAPI'),
    ('implied data-only restore', 'мається на увазі відновлення лише даних'),
    ('Access was denied by the user or server.', 'Відмовлено в доступі користувачем або сервером.'),
    ('Check to make the object insensitive (not selectable by mouse)', "Зробити цей об'єкт нечутливим до позначення"),
    ('The shortcut keys for turning Unicode typing on or off', 'Клавіатурне скорочення для вмикання і вимикання введення символів Unicode'),
    ('Child transition running', "Виконання переходу між дочірніми об'єктами"),
    ('EXIT cannot be used outside a loop, unless it has a label', 'EXIT можна використовувати поза циклом, тільки з зазначенням мітки'),
    ('Disc is not an Audio CD.', 'Диск не записано у форматі Audio CD.'),
    ('LDAP authentication method not supported', 'Підтримки методу розпізнавання LDAP не передбачено'),
    ('Select the virtual machine only by its ID', 'Вибрати віртуальну машину лише за її ідентифікатором'),
    ('Change the number of workspaces of the screen to NUMBER', 'Змінити кількість робочих областей на вказану КІЛЬКІСТЬ'),
    ('The unlock password was incorrect', 'Неправильний пароль для розблоковування'),
    ('shortcut window\x04Toggle Image Properties', 'Увімкнути або вимкнути показ панелі властивостей зображень'),
    ('Authentication is required to change your own user password', 'Для зміни пароля вашого користувача слід пройти розпізнавання'),
    ('Select window to share with the requesting app', 'Виберіть вікно, дані з якого слід оприлюднити за допомогою відповідної програми'),
    ('Please make a selection from the above', 'Будь ласка, виберіть один з наведених вище варіантів'),
    ("Yemen, Democratic, People's Democratic Republic of", 'Народна Демократична Республіка Ємен'),
    ('Name of attribute that is used as object class for sudo rules', "Назва атрибута, який використано як клас об'єктів для правил sudo"),
    ('envp not set by the security policy', 'envp не встановлено правилами захисту'),
    ('setend use is deprecated for ARMv8', 'використання setend є застарілим для ARMv8'),
    ('Zaza; Dimili; Dimli; Kirdki; Kirmanjki; Zazaki', 'заза; дімілі; дімлі; кірдкі; кірманджикі; зазакі'),
    ('Whether bold text should use the same color as normal text', 'Чи текст з жирним накресленням виводиться тим самим кольором, що й звичайний текст'),
    ('virtual def operand missing for statement', 'віртуальний операнд визначення відсутній для оператора'),
    ('Go online to set up Enterprise Login.', 'Увійдіть до інтернету, щоб налаштувати комерційний вхід.'),
    ('Built-in helper, rename not supported.', 'Вбудований допоміжний засіб, підтримки перейменовування не передбачено.'),
    ('Color scheme using Solarized dark color palette', 'Кольорова схема, що використовує палітру темних засвічених кольорів'),
    ('Print to Test Printer', 'Друкувати для випробування принтера'),
    ('OpenType layout\x04Vertical Alternates for Rotation', 'Вертикальні альтернативи для обертання'),
    ('Clear Recent List', 'Спорожнити список нещодавніх записів'),
    ('ActionsToolbar|Decrypt\x04Unlock/Open selected device', 'Розблокувати або відкрити позначений пристрій'),
    ('The URI of the web service that allows the calibration tools to upload a specific profile to the Internet.', 'Інтернет-адреса служби, за якою інструменти калібрування можуть вивантажити певний профіль.'),
    ('Could not parse invalid or corrupted data.', 'Не вдалось розібрати некоректні чи пошкоджені дані.'),
    ('None of the specified keys are writable', 'Серед вказаних ключів немає записуваних'),
    ('-n, --rcs                     output an RCS format diff', '-n, --rcs                     виводити у форматі diff систем керування версіями'),
    ('specify a module or tokens to remote', 'вказати модуль або жетони для remote'),
    ('Redirect Type of Service and Network', 'Тип переспрямовування служби і мережі'),
    ('Missing both car and cdr values from pair in XML file', 'Відсутні ключ і значення для пари у файлі XML'),
    ('Failed to start Software', 'Не вдалося запустити «Програмні засоби»'),
    ('Default location for the “Take Screenshot” dialogs. Default is the Screenshots directory.', 'Типова адреса для «Зробити знімок…». Типово — каталог знімків вікон.'),
    ('The contents are locked. In order to view the contents, enter the correct password.', 'Вміст заблоковано. Введіть правильний пароль, щоб переглянути вміст.'),
    ('Dump reserved and extra data', 'Створити дамп зарезервованих та додаткових даних'),
    ('GNU General Public License, version 3 or later', 'Загальна громадська ліцензія GNU (GNU GPL), версія 3 або новіша'),
    ('cannot start debugger; debugging mode disabled', 'не вдалося запустити засіб діагностики: режим діагностування вимкнено'),
    ('Shall the new role be allowed to create databases?', 'Чи дозволено новій ролі створювати бази даних?'),
    ('cannot load kernel symbols', 'не вдалося завантажити символи ядра'),
    ('Do not accept a key as being pressed unless held for @delay milliseconds.', 'Не приймати клавіш, як натиснений, поки він не утримувався впродовж @delay мілісекунд.'),
    ('History for the looking glass dialog', 'Історія для пошуку дзеркальних діалогів'),
    ('Logically impossible section reached in getftp()', 'У getftp() виявлено логічно неможливий розділ'),
    ('Global IPv6 forwarding is enabled in configuration, but not currently enabled in kernel', 'Маршрутизація IPv6 дозволена в конфігурації, але заборонена в ядрі'),
    ('Computers need to be set up for remote desktop before you can connect to them.', "Перш ніж ви зможете встановлювати з'єднання із віддаленими комп'ютерами, їх слід слід належно налаштувати."),
    ('Contacts will also integrate with online address books and automatically link contacts from different online sources.', 'Контакти також інтегровані з мережевими адресними книгами і автоматично сполучать контакти з різних джерел.'),
    ('3D acceleration for some of the supported operating systems', 'Прискорення просторової графіки для деяких із підтримуваних операційних систем'),
    ('Port to listen to', 'Порт, на якому слід очікувати на дані'),
    ('read exclude patterns from the VCS ignore files', 'прочитати взірці виключення із файлів ігнорування системи керування версіями'),
    ('File names should not end with a space', 'Назви файлів не мають закінчуватись пропуском'),
    ('Tobagonian Creole English', 'тобагонійська креольська англійська'),
    ('Name of the input method module used by GTK+.', 'Назва модулю методу введення, що використовується GTK+.'),
    ('Provide a facility for quickly viewing different kinds of files', 'Надає можливість швидко переглядати вміст файлів різних типів'),
    ('data checksums are already disabled in cluster', 'контрольні суми вже неактивовані в кластері'),
    ('Could not open audio device for playback. This version of the Open Sound System is not supported by this element.', 'Не вдалося відкрити пристрій для відтворення. Ця версія Open Sound System не підтримується цим елементом.'),
    ('Traces some select OpenGL calls', 'Маршрут деяких вибраних викликів OpenGL'),
    ("Authentication is needed to run `$(program)' as the super user", 'Для запуску «$(program)» від імені суперкористувача слід пройти розпізнавання'),
    ('Usage: @GPGCONF@ [options] (-h for help)', 'Використання: @GPGCONF@ [параметри] (-h — довідка)'),
    ('Invalid verify-x509-name.', 'Некоректне значення verify-x509-name.'),
    ('South Georgia and the South Sandwich Islands', 'Південна Георгія і Південні Сандвічеві Острови'),
    ('Convert language specific digits to ASCII digits', 'Перетворювати специфічні для мови цифри на цифри ASCII'),
    ('Place your left middle finger on the fingerprint reader', 'Прикладіть ваш лівий середній палець до пристрою для зчитування'),
    ('No headers, assuming HTTP/0.9', 'Відсутні заголовки, припускається, що це HTTP/0.9'),
    ('Show options to list windows or workspaces', 'Показати параметри списку вікон чи робочих областей'),
    ('Show process “CPU Time” column on startup', 'Показувати під час запуску стовпчик часу процесора'),
    ('No Standard User Accounts', 'Немає стандартних облікових записів користувачів'),
    ('Search for music albums (--all has no effect on this)', 'Пошук музичних альбомів (--all не впливає)'),
    ('could not acquire SSPI credentials', 'не вдалось отримати облікові дані SSPI'),
    ('change group to have given name', 'змінити групу так, щоб вона мала вказану назву'),
    ('The output below may help determine the cause of the error:', 'Наведені нижче дані можуть допомогти у пошуку причини помилки:'),
    ('Password has been already used. Choose another.', 'Цей пароль вже використано. Виберіть інший.'),
    ('expire the password for the named account (root only)', 'завершити строк дії пароля до облікового запису (лише root)'),
    ('could not create temporary file whilst writing archive', 'не вдалося створити тимчасовий файл під час запису архіву'),
    ("Use 'continue' command to quit the debugger and get back to the main menu", 'Для виходу з режиму діагностики і повернення до головного меню скористайтеся командою «continue»'),
    ('cannot specify a database name with --all', 'не можна вказати назву бази даних з --all'),
    ('Display option defaults in message', 'Показати типові значення параметрів у повідомленні'),
    ('Compatible with various PPTP servers including Microsoft.', 'Сумісний із різноманітними серверами PPTP, зокрема серверами Microsoft.'),
    ('Listen on loopback only', 'Очікувати дані лише на петльовому інтерфейсі'),
    ('Reduces the screen brightness when the computer is inactive.', "Зменшує яскравість екрана, якщо комп'ютер є бездіяльним."),
    ('Manage your contacts', 'Керування вашими записами контактів'),
    ('Provençal, Old (to 1500)', 'давньопровансальська (до 1500 року)'),
    ('A character string containing the name of the image creator, encoded in UTF-16LE.', "Рядок символів, що містить ім'я творця зображення, закодований у UTF-16LE."),
    ('Some directories are not accessible by authselect!', 'Деякі каталоги недоступні для authselect!'),
    ("'fqdn-no-update' and 'fqdn-serv-update' flags cannot be set at the same time", 'не можна одночасно встановлювати прапорці «fqdn-no-update» і «fqdn-serv-update»'),
    ('Invalid preceding regular expression', 'Некоректний попередній формальний вираз'),
    ('Set the temporary directory', 'Встановити адресу тимчасового каталогу'),
    ('Dark color scheme used in the Kate text editor', 'Темна схема кольорів з текстового редактора Kate'),
    ('The file system can only be resized to this size by converting to FAT16.', 'Змінити розмір файлової системи на вказаний розмір можна лише при перетворенні на FAT16.'),
    ('it is based on a (reversed) dictionary word', 'заснований на (записаному у зворотному порядку) слові зі словника'),
    ('debug_tag_type: extra tag attempted', 'debug_tag_type: випробуваний додатковий тег'),
    ('Copying user relation files', 'Копіювання файлів користувацьких відношень'),
    ('If this is set to true, when performing a drag and drop operation the hovered folder will open automatically after a timeout.', 'Якщо вказано, то операції перетягування на наведеній теці автоматично відкриватимуть її після певного часу.'),
    ('unknown libgphoto2 using program', 'невідома програма, що використовує libgphoto2'),
    ('The file has been changed by another program.', 'Сторонньою програмою було внесено зміни до файла.'),
    ("has characters other than ASCII alphanumerics, '-' or '+'", 'містить символи, які не є літерами або цифрами ASCII, символами «-» або «+»'),
    ('Mislabeled files exist', 'Виявлено файли з помилковими мітками'),
    ("warning: syntax error, expected ';' after string", "попередження: синтаксична помилка, після рядка очікувався знак ';'"),
    ('I/O plugin error', 'Помилка у додатку введення-виведення'),
    ("implementation error: tar files can't have more than one open file", 'помилка реалізації: файли tar не можуть мати більше одного відкритого файлу'),
    ('Could not query kernel driver of device.', 'Не вдалося опитати драйвер ядра для пристрою.'),
    ('could not create thread for alarm', 'не вдалося створити потік для сигналізації'),
    ('Failed to get fragment URL.', 'Не вдалося отримати адреси фрагмента.'),
    ('Check folder sizes and available disk space', 'Перевірте розміри тек та дисковий простір'),
    ('math symbol\x04does not contain as normal subgroup or equal', 'не містить як нормальну підгрупу і не дорівнює'),
    ('selecting default shared_buffers ... ', 'обирається значення shared_buffers... '),
    ('The environment variable FIND_BLOCK_SIZE is not supported, the only thing that affects the block size is the POSIXLY_CORRECT environment variable', 'Змінна оточення FIND_BLOCK_SIZE не підтримується, на розмір блоку впливає лише змінна оточення POSIXLY_CORRECT'),
    ('Invalid textual address form', 'Некоректний текстовий формат адреси'),
    ('could not parse start LSN', 'не вдалося проаналізувати початковий LSN'),
    ('Override standard autostart directories', 'Перевизначити стандартні каталоги автозапуску'),
    ('Pseudodirective .loc is only valid when generating ELF', 'Псевдодиректива .loc є чинною лише під час створення ELF'),
    ("exceeded PCRE's line length limit", 'перевищено обмеження на довжину рядка PCRE'),
    ('No XML schema associated with this config object', 'З цим об’єктом налаштування не пов’язано жодної схеми XML'),
    ('codec the audio data is stored in', 'кодек, яким закодовані звукові дані'),
    ('Authentication is required to run a program as another user', 'Для виконання програми від імені іншого користувача слід пройти розпізнавання'),
    ('Invalid mute specification', 'Некоректна специфікація вимикання звуку'),
    ('Launch web remote control', 'Запустити віддалене інтернет-керування'),
    ('Install mode to a specific prefix', 'Встановити режим до вказаного префіксом каталогу'),
    ('Failed to discover realm. See diagnostics.', 'Не вдалося виявити область. Див. діагностичні повідомлення.'),
    ('The width of the video captured from the camera, in pixels', 'Ширина відеозапису, одержаного з камери, в пікселях'),
    ('must specify archive location', 'необхідно вказати розташування архіва'),
    ('Choose files to send', 'Виберіть файли, які потрібно надіслати'),
    ('Fast GPU accelerated image rendering', 'Швидкої обробка зображення за допомогою графічного процесора'),
    ('BMP image has bogus header data', 'Зображення формату BMP містить неправильні дані в заголовку'),
    ('Timeline IDs must be in increasing sequence.', 'Ідентифікатори ліній часу повинні збільшуватись.'),
    ('Credentials have expired', 'Строк дії реєстраційних даних вичерпано'),
    ("migrate yum's history, group and yumdb data to dnf", 'перенести журнал yum, дані щодо груп та yumdb до dnf'),
    ('Invalid preceding regular expression', 'Помилка у попередньому формальному виразі'),
    ('South American Indian languages', 'Південноамериканські індіанські мови'),
    ('It makes it easy to find the documentation you need, with interactive search and bookmarks.', 'Із цією програмою вам буде просто знайти потрібну документацію — передбачено інтерактивний пошук та закладки.'),
    ('Color scheme using Solarized dark color palette', 'Кольорова схема, що використовує палітру темних засвічених кольорів'),
    ('permissions error in pam_timestamp_check', 'помилка прав доступу у pam_timestamp_check'),
    ('Malformed regular expression', 'Регулярний вираз сформовано неправильно'),
    ('The font family used as the default for content using monospace font.', 'Гарнітура шрифту, яку буде типово використано для даних із моноширинним шрифтом.'),
    ('Turn geolocation support on and off.', 'Увімкнути/вимкнути підтримування географічного розміщення.'),
    ('View and manage system resources', 'Переглядати та керувати системними ресурсами'),
    ('Install additional software from an NFS server', 'Встановити додаткові програми з сервера NFS'),
    ('Input file descriptor is NULL.', 'Файловий дескриптор вводу дорівнює NULL.'),
    ('Selected plan _APN (Access Point Name):', 'Виберіть _точку доступу (APN) цього тарифного плану:'),
    ('Switch to Full Width Punctuation Mode', 'Перемкнутися на режим повноширинної пунктуації'),
    ('duplicate shadow password entry', 'дублювання запису у файлі прихованих паролів'),
    ('Password hint\x04This is a weak password. Try to use more numbers.', 'Цей пароль — слабкий. Намагайтесь уживати якнайбільше чисел.'),
    ('Save/restore indicators together with layout groups', 'Зберігати/відновлювати індикатори разом з групами розкладок'),
    ('Clocks for world times, plus alarms, stopwatch and a timer', 'Годинники для перегляду часу, а також будильники, секундомір і таймер'),
    ('No supported ECC curves were found', 'Не виявлено підтримуваних кривих ECC'),
    ('List content with file details', 'Вивести список вмісту із подробицями щодо файлів'),
    ('Process the Java exception using the Red Hat infrastructure', 'Обробити дані щодо виключення Java за допомогою інфраструктури Red Hat'),
    ('Full search_contacts are not stored in cache. vcards cannot be returned.', 'Повні дані search_contact не зберігаються в кеші. Неможливо повернути дані vcards.'),
    ('South American Indian languages', 'Південноамериканські індіанські мови'),
    ('Reload with Apple Pay', 'Перезавантажити за допомогою Apple Pay'),
    ('The server did not accept the WebSocket handshake.', 'Сервер не прийняв рукостискання WebSocket.'),
    ('Requested PBKDF type is not supported for LUKS1.', 'Підтримки бажаного типу PBKDF для LUKS1 не передбачено.'),
    ('Authentication is required to send the entered passphrase back to the system.', 'Для надсилання введеного пароля до системи слід пройти розпізнавання.'),
    ('check of access.conf during account authorization', 'перевіряти access.conf під час уповноваження облікових записів'),
    ('unpaired UTF-8 bidirectional control character detected', 'виявлено символ двобічного керування UTF-8 без відповідника'),
    ('failed to unlink scratch file', 'не вдалося скасувати символічне посилання на тимчасовий файл'),
    ('Error creating file object', "Помилка при створенні об'єкту файлу"),
    ('LZO adaptive (legacy)', 'Адаптивний LZO (застарілий варіант)'),
    ('Disable misfeatures that are required by old or broken applications', 'Вимкнути неправильні функції, що потрібні лише старим або пошкодженим програмам'),
    ('No discriminatory language of any kind', 'Без дискримінацій в будь-яких її проявах'),
    ('Action on title bar middle-click', 'Дія при клацанні середньою кнопкою на заголовку вікна'),
    ('Once installed you will need to restart this app.', 'Після встановлення вам слід перезапустити цю програму.'),
    ('Show flags in the applet to indicate the current layout', 'Показувати прапори у аплеті для індикації теперішньої розкладки'),
    ("System policy prevents sending or manipulating this device's text messages.", 'Правила системи забороняють надсилання або керування текстовими повідомленнями цього пристрою.'),
    ('You can change the order on language bar', 'Ви можете змінити порядок на панелі мов'),
    ('Uruguay Peso en Unidades Indexadas (UI)', 'Одиниця індексування уругвайського песо (UI)'),
    ('Couldn’t parse public SSH key', 'Неможливо розібрати відкритий ключ SSH'),
    ('View and use virtual machines', 'Перегляд і використання віртуальних машин'),
    ('Toggle Sidebar', 'Увімкнути або вимкнути бічну панель'),
    ('Unmatched [, [^, [:, [., or [=', 'Вираз без парних [, [^, [:, [. або [='),
    ("[deprecated, use repoquery --deplist] List package's dependencies and what packages provide them", '[застарілий, користуйтеся repoquery --deplist] Показати список залежностей пакунка та пакунки, які їх надають'),
    ('Floating point exception', 'Помилка операції з крапкою, що плаває'),
    ('A source providing a list of subtitles for a video', 'Джерело, яка надає список субтитрів до відео'),
    ('Whether MediaSource should be enabled.', 'Визначає, чи слід вмикати MediaSource.'),
    ('Category of AudioVideo\x04Audio Creation & Editing', 'Створення і редагування звукових даних'),
    ('no start WAL location given', 'не задано початкове розташування WAL'),
    ('Select the spell checking _language.', 'Виберіть _мову перевірки правопису.'),
    ('App is fully sandboxed', 'Програма повністю працює у «пісочниці»'),
    ('Text banner message to show in the login window.', 'Текст повідомлення заголовка, що буде показано у вікні входу.'),
    ('Add new events to this calendar by default.', 'Типово додати нову подію до цього календаря.'),
    ('Send email and manage your schedule', 'Робота з електронною поштою та особистим розкладом'),
    ('Manages encrypted volume keys and passphrases.', 'Керування ключами шифрування томів і паролями.'),
    ('Password generation failed - required entropy too low for settings', 'Спроба створення пароля зазнала невдачі: рівень ентропії є занизьким'),
    ('Set the maximum cache size to SIZE bytes', 'Встановити для максимального розміру кешу у байтах значення РОЗМІР'),
    ('Message is already in session queue', 'Повідомлення вже перебуває у черзі сеансу'),
    ('Applications and sites saved from Web', 'Програми і сайти, збережені з інтернету'),
    ('Failed to read from stdin', 'Не вдалося виконати читання з stdin'),
    ('Remove the selected mount point(s).', 'Вилучити позначені точки монтування.'),
    ("Can't add host USB device: USB is disabled in this host", 'Не вдалося додати пристрій USB основної системи: у цій основній системі вимкнено USB'),
    ('The document contains only empty pages', 'Документ містить лише порожні сторінки'),
    ('print this message and exit', 'показати це повідомлення і завершити роботу'),
    ('Privileges are required to enable/disable a printer, or a class.', 'Для вмикання або вимикання принтера або класу потрібні відповідні права доступу.'),
    ('Libvirt did not detect any UEFI/OVMF firmware image installed on the host.', 'Libvirt не вдалося виявити жодного образу мікропрограми UEFI/OVMF, встановленого у основній системі.'),
    ('Next node in node reading order', 'Наступний вузол за порядком читання вузлів'),
    ('reading column info for interesting tables', 'читання інформації про стовпці цікавлячої таблиці'),
    ('Use custom theme name for language panel', 'Використовувати нетипову тему для мовної панелі'),
    ("If set to 'true' the VNC backend will be initialized.", 'Якщо встановлено у значення «true» модуль VNC буде ініціалізовано.'),
    ('force NAME as group for added files', 'встановлення групи з вказаною назвою групою власника доданих файлів'),
    ('Symmetric key callback not provided', 'Не вказано зворотного виклику симетричного ключа'),
    ('Polish (Germany, no dead keys)', 'Польська (Німеччина, без сліпих клавіш)'),
    ('Backup file creation failed', 'Помилка при створенні резервної копії файлу'),
    ('Audio Gateway (A2DP Source & HSP/HFP AG)', 'Звуковий шлюз (джерело A2DP і HSP/HFP AG)'),
    ('The current position does not hold a string type', 'Поточна позиція не утримує тип рядка'),
    ('Modulus division is only defined for integers', 'Ділення за модулем визначено лише для цілих'),
    ('Select a priority for module operations', 'Виберіть пріоритетність для дій з модулями'),
    ('Transformed PNG not RGB or RGBA.', 'Перетворене зображення формату PNG не має тип RGB чи RGBA.'),
    ('Epson L Photo Paper (tear-off borders)', 'Фотопапір Epson L (з фігурними краями)'),
    ('Model column to search through during interactive search', 'Стовпчик моделі, за яким слід виконати інтерактивний пошук в міру набору'),
    ('Compatible with Cisco VPN concentrators configured to use IPsec.', 'Сумісний із концентраторами VPN Cisco, які налаштовано на використання IPsec.'),
    ('Undo the last change in the image', 'Повернути останню зміну в зображенні'),
    ('No driver for this printer.', 'Для цього принтера відсутній драйвер.'),
    ('Refresh defined repository index services.', 'Оновити вказані сервіси індексу сховища.'),
    ('-l | -r | [-s] grubdev osdisk.', '-l | -r | [-s] пристрій_grub диск_ОС.'),
    ('Print all key/value pairs in a directory.', 'Вивести усі пари ключ/значення в каталозі.'),
    ('Swipe your right ring finger across the fingerprint reader', 'Проведіть вашим правим безіменним пальцем вздовж пристрою для зчитування'),
    ('module control-line cannot be in included file', 'рядок керування модулем не може зберігатися у включеному файлі'),
    ('Characters will appear here if you use them', "Символи з'являтимуться тут у міру їхнього використання"),
    ('#  Recipe currently running (THIS IS A BUG).', '#  Виконується обробка (ЦЕ ПОМИЛКА)'),
    ('Enter to keep the current selection[+], or type selection number: ', 'Enter - зберегти поточний вибір[+], або вкажіть номер: '),
    ('Software Installation Restrictions', 'Обмеження на встановлення програмного забезпечення'),
    ('Authentication is required to remove an entry from /etc/fstab file', 'Щоб отримати доступ до вилучення запису з файла /etc/fstab, слід пройти розпізнавання'),
    ('set the rate in characters per second.', 'встановити швидкість у символах за секунду.'),
    ('South Georgia and the South Sandwich Islands', 'Південна Джорджія та Південні Сандвічеві острови'),
    ('TLS Connection does not support TLS-Exporter feature', "У з'єднанні TLS не передбачено підтримки можливості TLS-Exporter"),
    ('Please try a different file extension like .png or .jpg.', 'Спробуйте різні розширення файлів, наприклад .png або .jpg.'),
    ('The device sensor should have been cleaned prior to scanning and the output file resolution should be at least 200dpi.', 'Перед скануванням треба очистити сенсор пристрою, а роздільність кінцевого файла має бути не менша ніж 200dpi.'),
    ('Remote data does not contain valid identifier', 'Віддалені дані не містять коректного ідентифікатора'),
    ('JSON data must be UTF-8 encoded', 'Дані JSON повинні бути у кодуванні UTF-8'),
    ('Number of CRC errors during UDMA mode', 'Кількість помилок CRC при роботі в UDMA режимі'),
    ('Do not copy DT_NEEDED links mentioned inside DSOs that follow', 'не копіювати посилання DT_NEEDED, згадані у DSO, вказаних надалі'),
    ('Can’t import non-socket as SoupSocket', 'Неможливо імпортувати об’єкти відмінні від сокета як SoupSocket'),
    ('Bad argument to system call', 'Неправильний аргумент у системному виклику'),
    ('Cannot export VPN connection', "Не вдається імпортувати з'єднання VPN"),
    ('Allocation failure for string from stdin', "Не вдалося розмістити у пам'яті рядок зі стандартного джерела вхідних даних (stdin)"),
    ('must specify oldest kept WAL file', 'необхідно вказати найдавніший збережений WAL-файл'),
    ('Force calibration ignoring all and any calibration caches', 'Примусове калібрування з ігноруванням усіх кешованих даних калібрування'),
    ('United Kingdom of Great Britain and Northern Ireland', 'Об’єднане Королівство Великої Британії та Північної Ірландії'),
    ('GCredentials is not implemented on this OS', 'Тип GCredentials не реалізовано для цієї ОС'),
    ('OpenCV failed to load template image', 'OpenCV не вдалося завантажити зображення шаблону'),
    ('Select devices to share with the requesting application', 'Виберіть пристрої, які слід оприлюднити за допомогою відповідної програми'),
    ('Throttle problem directory creation to 1 per INT second', 'Встановлення інтервалу створення каталогу проблеми у значення від 1 до INT секунд'),
    ('Spell checker error: no language set. It’s maybe because no dictionaries are installed.', 'Помилка перевірки правопису: не встановлено жодної мови. Можливо, тому, що не встановлено жодних словників.'),
    ('Cannot issue command, no stream available', 'Не вдалося видати команду, немає доступного потоку'),
]

print("у запасному корпусі пар:", len(MINI_CORPUS))

In [ ]:
if len(system_pairs) >= 1000:
    corpus = system_pairs
    CORPUS_NAME = "системні .mo-каталоги"
else:
    corpus = MINI_CORPUS
    CORPUS_NAME = "вбудований мінікорпус"
    print("!" * 70)
    print("УВАГА: української локалі в системі немає, працюємо на мінікорпусі.")
    print("Числа будуть іншими, ніж у лекції, — це очікувано.")
    print("!" * 70)

english = [pair[0] for pair in corpus]
ukrainian = [pair[1] for pair in corpus]

print("джерело корпусу:", CORPUS_NAME)
print("документів:      ", len(corpus))
print()
print("приклад пари:")
print("  англ:", english[0])
print("  укр: ", ukrainian[0])

### Скільки в корпусі слів і символів

Два прості лічильники, які знадобляться далі скрізь. Слово тут — те, що відділене
пробілом; так рахують і в лекції.

Зверни увагу на останній рядок: **слів в обох мовах майже порівну**, а **символів
українською помітно більше**. Українське слово просто довше — і це буде окремий
розділ.

In [ ]:
english_words = sum(len(text.split()) for text in english)
ukrainian_words = sum(len(text.split()) for text in ukrainian)
english_chars = sum(len(text) for text in english)
ukrainian_chars = sum(len(text) for text in ukrainian)

print("слововживань англійською:", english_words)
print("слововживань українською:", ukrainian_words)
print("символів англійською:    ", english_chars)
print("символів українською:    ", ukrainian_chars)
print()
print("слів укр / слів англ:      %.4f" % (ukrainian_words / english_words))
print("символів укр / символів англ: %.4f" % (ukrainian_chars / english_chars))
print()
print("символів на слово, англ: %.4f" % (english_chars / english_words))
print("символів на слово, укр:  %.4f" % (ukrainian_chars / ukrainian_words))
print("відношення довжин слів:  %.4f" % ((ukrainian_chars / ukrainian_words) / (english_chars / english_words)))

## 2 · Службові функції

Далі все тримається на трьох речах: **як ділити корпус на навчальну й тестову
частини**, **як навчити токенізатор** і **як порахувати ціну тексту в токенах**.
Оголосимо їх один раз.

Чому взагалі потрібна тестова частина: токенізатор, якого питають про той самий
текст, на якому його вчили, виглядає кращим, ніж він є. Особливо це псує замір
**невідомих слів** — на навчальному тексті їх не буває за побудовою.

In [ ]:
WORD_SPLIT = pre_tokenizers.Whitespace()  # ріже по пробілах і відділяє розділові знаки

TRAIN_SHARE = 0.8
SEEDS = (0, 1, 2)


SPLIT_CACHE = {}


def split_corpus(seed):
    """Ділить корпус на навчальну й тестову частини. Зерно міняє саме поділ."""
    if seed in SPLIT_CACHE:
        return SPLIT_CACHE[seed]
    order = list(range(len(corpus)))
    random.Random(seed).shuffle(order)
    cut = int(TRAIN_SHARE * len(order))
    train_ids, test_ids = order[:cut], order[cut:]
    SPLIT_CACHE[seed] = (
        [english[i] for i in train_ids], [ukrainian[i] for i in train_ids],
        [english[i] for i in test_ids], [ukrainian[i] for i in test_ids],
    )
    return SPLIT_CACHE[seed]


def train_bpe(texts, vocab_size, extra_specials=()):
    """Навчає BPE-токенізатор із нуля на переданих текстах."""
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = WORD_SPLIT
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"] + list(extra_specials),
        show_progress=False,
    )
    tokenizer.train_from_iterator(texts, trainer)
    return tokenizer


TOKENIZER_CACHE = {}


def get_tokenizer(language, vocab_size, seed):
    """Навчений BPE для мови й зерна — з кешем.

    Той самий токенізатор потрібен у кількох розділах, а навчання коштує секунди.
    Кеш робить зошит удвічі коротшим за часом і нічого не міняє в числах.
    """
    key = (language, vocab_size, seed)
    if key not in TOKENIZER_CACHE:
        english_train, ukrainian_train, _, _ = split_corpus(seed)
        train_texts = english_train if language == "англ" else ukrainian_train
        TOKENIZER_CACHE[key] = train_bpe(train_texts, vocab_size)
    return TOKENIZER_CACHE[key]


def train_wordpiece(texts, vocab_size):
    """Те саме, але алгоритм WordPiece — той, на якому стоїть BERT."""
    tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]", max_input_chars_per_word=200))
    tokenizer.pre_tokenizer = WORD_SPLIT
    trainer = trainers.WordPieceTrainer(
        vocab_size=vocab_size, special_tokens=["[UNK]"], show_progress=False)
    tokenizer.train_from_iterator(texts, trainer)
    return tokenizer


def token_cost(tokenizer, texts):
    """Ціна тексту в токенах: на тисячу слів і на тисячу символів, за один прохід.

    Два числа рахуємо разом навмисно: кодування великого корпусу — найдорожча
    операція зошита, і робити її двічі заради двох дільників немає сенсу.
    """
    encoded = tokenizer.encode_batch(texts)
    total_tokens = sum(len(item.ids) for item in encoded)
    total_words = sum(len(text.split()) for text in texts)
    total_chars = sum(len(text) for text in texts)
    return 1000.0 * total_tokens / total_words, 1000.0 * total_tokens / total_chars


def tokens_per_1000_words(tokenizer, texts):
    """Коротка форма, коли потрібне лише перше з двох чисел."""
    return token_cost(tokenizer, texts)[0]


def mean_and_spread(values):
    """Середнє й розкид по зернах. Розкид — стандартне відхилення по всій вибірці."""
    return statistics.mean(values), statistics.pstdev(values)


print("службові функції готові, зерен:", len(SEEDS))
print("перевірка поділу:", [len(part) for part in split_corpus(0)])
print("кеш токенізаторів порожній:", len(TOKENIZER_CACHE) == 0)

## 3 · Три рівні різання: символи, слова, субслова

Текст можна різати трьома способами, і вибір між ними — це вибір між двома бідами:

| рівень | словник | невідомі слова |
|---|---|---|
| символи | крихітний | не буває |
| слова | велетенський | багато |
| субслова | середній | не буває |

Перевіримо це числами. Міряємо дві речі на **тестовій** частині, якої токенізатор
не бачив:

- **розмір словника** — скільки різних одиниць треба тримати;
- **частка невідомих (OOV, out-of-vocabulary)** — яка частка слів тестового тексту
  не має рядка в словнику.

Для рівня «слова» беремо два варіанти: усі словоформи навчальної частини й **лише
чотири тисячі найчастіших** — бо словник моделі завжди обмежений, і чесно порівнювати
з субсловами треба саме при однаковому розмірі.

In [ ]:
LEVELS_VOCAB = 4000  # однаковий бюджет для «топ-слів» і для BPE

levels = collections.defaultdict(list)

for seed in SEEDS:
    english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(seed)
    for language, train_texts, test_texts in (
        ("англ", english_train, english_test),
        ("укр", ukrainian_train, ukrainian_test),
    ):
        # рівень 1: символи
        train_chars = set("".join(train_texts))
        test_chars = "".join(test_texts)
        char_oov = sum(1 for c in test_chars if c not in train_chars) / len(test_chars)

        # рівень 2: слова
        word_counts = collections.Counter()
        for text in train_texts:
            word_counts.update(word for word, _ in WORD_SPLIT.pre_tokenize_str(text))
        test_words = []
        for text in test_texts:
            test_words.extend(word for word, _ in WORD_SPLIT.pre_tokenize_str(text))
        word_oov = sum(1 for w in test_words if w not in word_counts) / len(test_words)
        top_words = set(w for w, _ in word_counts.most_common(LEVELS_VOCAB))
        top_oov = sum(1 for w in test_words if w not in top_words) / len(test_words)

        # рівень 3: субслова
        subword_tokenizer = get_tokenizer(language, LEVELS_VOCAB, seed)
        unknown_id = subword_tokenizer.token_to_id("[UNK]")
        encoded = subword_tokenizer.encode_batch(test_texts)
        total = sum(len(item.ids) for item in encoded)
        subword_oov = sum(item.ids.count(unknown_id) for item in encoded) / total

        levels[(language, "символи")].append((len(train_chars), char_oov))
        levels[(language, "слова, усі")].append((len(word_counts), word_oov))
        levels[(language, "слова, топ-4000")].append((LEVELS_VOCAB, top_oov))
        levels[(language, "субслова, BPE-4000")].append((LEVELS_VOCAB, subword_oov))

print("%-6s %-20s %10s %14s" % ("мова", "рівень", "словник", "невідомих"))
for (language, level), rows in levels.items():
    vocab_mean, _ = mean_and_spread([r[0] for r in rows])
    oov_mean, oov_spread = mean_and_spread([r[1] for r in rows])
    print("%-6s %-20s %10.0f %11.4f ± %.4f" % (language, level, vocab_mean, oov_mean, oov_spread))

### Що тут видно

**Словник зі слів для української в півтора раза більший, ніж для англійської** —
на тому самому змісті. Це не про багатство лексики: те саме поняття просто
розсипається на десяток словоформ.

**І при цьому він гірший.** Дірка в словнику (частка невідомих слів) для української
теж більша. А коли обом мовам дати однаковий бюджет у чотири тисячі рядків,
українська втрачає вдвічі більше слів, ніж англійська.

Субслова закривають дірку повністю: невідомим лишається хіба символ, якого в
навчальній частині не траплялось узагалі.

## 4 · Чому не досить порізати по пробілах

Перш ніж будувати щось складне, варто перевірити найпростіше: `text.split()`.
Для української цього мало з двох причин, і обидві можна побачити числом.

**Перша: розділові знаки приклеюються до слів.** `"файл,"` і `"файл"` стають різними
рядками словника.

**Друга, підступніша: апостроф.** В українських текстах його пишуть щонайменше
трьома різними символами Unicode, і для компʼютера це три різні слова.

In [ ]:
apostrophe_variants = "'\u02bc\u2019\u0060\u00b4\u2018"
all_ukrainian_text = " ".join(ukrainian)

print("як пишуть апостроф у цьому корпусі:")
counts = collections.Counter(c for c in all_ukrainian_text if c in apostrophe_variants)
for character, count in counts.most_common():
    print("   U+%04X  %-6s  %d разів" % (ord(character), repr(character), count))

# скільки словоформ злиється, якщо всі варіанти звести до одного символу
raw_forms = collections.Counter()
for text in ukrainian:
    raw_forms.update(word.lower() for word, _ in pre_tokenizers.WhitespaceSplit().pre_tokenize_str(text))


def unify_apostrophe(word):
    """Зводить усі написання апострофа до канонічного U+02BC."""
    for character in "'\u2019\u0060\u00b4\u2018":
        word = word.replace(character, "\u02bc")
    return word


unified_forms = collections.Counter()
for form, count in raw_forms.items():
    unified_forms[unify_apostrophe(form)] += count

with_apostrophe = [w for w in raw_forms if any(c in w for c in apostrophe_variants)]
print()
print("словоформ усього:                 ", len(raw_forms))
print("з них містять апостроф:           ", len(with_apostrophe))
print("словоформ після зведення до U+02BC:", len(unified_forms))
print("злилося в пари-двійники:          ", len(raw_forms) - len(unified_forms))

In [ ]:
sample = ukrainian[:5000]

naive_types = set()
for text in sample:
    naive_types.update(text.split())

careful_types = set()
for text in sample:
    careful_types.update(word for word, _ in WORD_SPLIT.pre_tokenize_str(text))

print("на", len(sample), "документах:")
print("  text.split() дає різних рядків:      ", len(naive_types))
print("  розбиття з відділенням розділових:   ", len(careful_types))
print("  зайвих рядків через приліплені знаки:", len(naive_types) - len(careful_types))
print()
example = "Не вдалося зʼєднатися з сервером, спробуй ще раз."
print("приклад:")
print("  split():   ", example.split())
print("  наш поділ: ", [word for word, _ in WORD_SPLIT.pre_tokenize_str(example)])

## 5 · BPE своїми руками

Алгоритм BPE (byte pair encoding, кодування парами байтів) вміщується в одне речення:

> **поки словник не набрався — знайди найчастішу пару сусідніх одиниць і злий її в
> одну.**

Починаємо з окремих символів. Кожне злиття додає у словник один новий рядок і
скорочує тексти. Усе.

Щоб це було видно очима, візьмімо шість справжніх слів корпусу з їхніми справжніми
частотами. Частоти беремо з корпусу, а не зі стелі.

In [ ]:
DEMO_WORDS = ["файл", "файлів", "каталог", "файли", "файлу", "каталогу"]

corpus_word_counts = collections.Counter()
for text in ukrainian:
    corpus_word_counts.update(word.lower() for word, _ in WORD_SPLIT.pre_tokenize_str(text))

demo_frequencies = {word: corpus_word_counts[word] for word in DEMO_WORDS}
for word, count in demo_frequencies.items():
    print("%-10s трапляється в корпусі %d разів" % (word, count))

### Один крок алгоритму

Кожне слово тримаємо як **список шматків**. На старті шматки — окремі літери.
Один крок робить три речі: порахувати всі сусідні пари, обрати найчастішу, злити її
в усіх словах одразу.

Разом із парою-переможцем друкуємо ще одне число — **скільки пар набрали ту саму
частоту**. Воно знадобиться за хвилину.

In [ ]:
def count_pairs(word_pieces, frequencies):
    """Скільки разів кожна пара сусідніх шматків трапляється в усьому корпусі."""
    pairs = collections.Counter()
    for word, pieces in word_pieces.items():
        for position in range(len(pieces) - 1):
            pairs[(pieces[position], pieces[position + 1])] += frequencies[word]
    return pairs


def apply_merge(word_pieces, pair):
    """Зливає задану пару в один шматок у кожному слові."""
    left, right = pair
    merged = {}
    for word, pieces in word_pieces.items():
        result = []
        position = 0
        while position < len(pieces):
            if position < len(pieces) - 1 and pieces[position] == left and pieces[position + 1] == right:
                result.append(left + right)
                position += 2
            else:
                result.append(pieces[position])
                position += 1
        merged[word] = result
    return merged


def learn_bpe_by_hand(frequencies, merge_limit):
    """Наш власний BPE. Повертає список злиттів і те, як поділені слова наприкінці."""
    word_pieces = {word: list(word) for word in frequencies}
    history = []
    for _ in range(merge_limit):
        pairs = count_pairs(word_pieces, frequencies)
        if not pairs:
            break
        best_pair, best_count = pairs.most_common(1)[0]
        # скільки ще пар мають рівно таку саму частоту — тобто наскільки вибір випадковий
        rivals = sum(1 for count in pairs.values() if count == best_count)
        word_pieces = apply_merge(word_pieces, best_pair)
        history.append((best_pair, best_count, rivals, {w: list(p) for w, p in word_pieces.items()}))
    return history, word_pieces


history, hand_pieces = learn_bpe_by_hand(demo_frequencies, merge_limit=14)

print("%-3s %-14s %8s %11s %8s" % ("№", "злиття", "частота", "претенденти", "токенів"))
for step, (pair, count, rivals, pieces) in enumerate(history, start=1):
    total_pieces = sum(len(p) for p in pieces.values())
    print("%-3d %-14s %8d %11d %8d" % (step, pair[0] + " + " + pair[1], count, rivals, total_pieces))

print()
print("як поділені слова після", len(history), "злиттів:")
for word, pieces in hand_pieces.items():
    print("   %-10s -> %s" % (word, " | ".join(pieces)))

### Тепер те саме бібліотекою

`tokenizers` навчить BPE на тому самому крихітному корпусі. Бюджет словника ставимо
рівно такий: **алфавіт плюс наші чотирнадцять злиттів**.

І тут на нас чекає перша чесна несподіванка.

In [ ]:
demo_texts = []
for word, count in demo_frequencies.items():
    demo_texts.extend([word] * count)

demo_alphabet = sorted(set("".join(demo_frequencies)))
library_demo = train_bpe(demo_texts, vocab_size=len(demo_alphabet) + len(history) + 1)

import json as _json
library_merges = [tuple(pair) for pair in _json.loads(library_demo.to_str())["model"]["merges"]]
our_merges = [pair for pair, _, _, _ in history]

print("алфавіт:", len(demo_alphabet), "символів")
print()
print("%-3s %-16s %-16s" % ("№", "наші злиття", "злиття бібліотеки"))
for step in range(len(our_merges)):
    ours = our_merges[step][0] + " + " + our_merges[step][1]
    theirs = library_merges[step][0] + " + " + library_merges[step][1]
    mark = "" if ours == theirs else "   <- різні"
    print("%-3d %-16s %-16s%s" % (step + 1, ours, theirs, mark))

### Списки злиттів **не збіглися** — і це не помилка

Подивись на колонку «претенденти» вище. На перших кроках найчастішу частоту мали
**три пари одразу**: `ф+а`, `а+й`, `й+л` — усі трапляються 5638 разів, бо стоять
усередині тих самих слів. Хто з них піде першим, алгоритм не визначає: це нічия,
і кожна бібліотека розводить її по-своєму.

Але результат від цього не залежить. Перевіримо не порядок злиттів, а **те, як
поділені слова** — і ось воно мусить збігтися символ у символ.

In [ ]:
library_pieces = {word: library_demo.encode(word).tokens for word in demo_frequencies}

print("%-10s %-22s %-22s" % ("слово", "наш поділ", "поділ бібліотеки"))
for word in demo_frequencies:
    print("%-10s %-22s %-22s" % (word, " | ".join(hand_pieces[word]), " | ".join(library_pieces[word])))

assert all(hand_pieces[word] == library_pieces[word] for word in demo_frequencies), \
    "наш BPE розійшовся з бібліотечним!"
print()
print("✅ збігається: наш власний BPE ділить слова так само, як tokenizers")

Це і є найцінніше, що дає ця частина: **всередині бібліотеки немає магії**.
Тридцять рядків рахують пари й зливають найчастішу — і виходить те саме.

Окремо зверни увагу на злиття №10 і №11: алгоритм зліпив спершу `файл+і`, потім
`файлі+в`, тобто **сам знайшов закінчення родового відмінка множини**. Жодного
уявлення про морфологію в нього немає — тільки лічильник пар.
[Тема 03](../03-morphology/lecture.html) покаже, що буває, коли морфологію знати
по-справжньому.

## 6 · Скільки коштує та сама думка двома мовами

Ось головний замір теми. Обидві мови кажуть **те саме** — у нас паралельний корпус.
Обидві дістають **свій власний** токенізатор, навчений на своїй мові. Питання одне:
скільки токенів коштує тисяча слів.

Шість розмірів словника, три зерна. Це найдовша клітинка зошита — близько
півтори хвилини.

In [ ]:
VOCAB_SIZES = [500, 1000, 2000, 4000, 8000, 16000]

sweep = {size: [] for size in VOCAB_SIZES}

for seed in SEEDS:
    english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(seed)
    for vocab_size in VOCAB_SIZES:
        english_words_cost, english_chars_cost = token_cost(
            get_tokenizer("англ", vocab_size, seed), english_test)
        ukrainian_words_cost, ukrainian_chars_cost = token_cost(
            get_tokenizer("укр", vocab_size, seed), ukrainian_test)
        sweep[vocab_size].append((english_words_cost, ukrainian_words_cost,
                                  english_chars_cost, ukrainian_chars_cost))
        # великі словники тримати в памʼяті далі не треба
        if vocab_size not in (4000, 8000):
            TOKENIZER_CACHE.pop(("англ", vocab_size, seed), None)
            TOKENIZER_CACHE.pop(("укр", vocab_size, seed), None)
    print("зерно", seed, "готове, минуло %.0f с" % (time.time() - STARTED_AT))

print()
print("%7s %18s %18s %16s" % ("словник", "англ / 1000 слів", "укр / 1000 слів", "націнка"))
markup_table = {}
for vocab_size in VOCAB_SIZES:
    rows = sweep[vocab_size]
    english_mean, english_spread = mean_and_spread([r[0] for r in rows])
    ukrainian_mean, ukrainian_spread = mean_and_spread([r[1] for r in rows])
    markups = [100.0 * (r[1] / r[0] - 1) for r in rows]
    markup_mean, markup_spread = mean_and_spread(markups)
    markup_table[vocab_size] = (english_mean, ukrainian_mean, markup_mean, markup_spread)
    print("%7d %11.1f ± %-4.1f %11.1f ± %-4.1f %9.1f %% ± %.2f" % (
        vocab_size, english_mean, english_spread,
        ukrainian_mean, ukrainian_spread, markup_mean, markup_spread))

first, last = markup_table[VOCAB_SIZES[0]][2], markup_table[VOCAB_SIZES[-1]][2]
print()
print("націнка впала з %.1f %% до %.1f %% — у %.2f раза" % (first, last, first / last))

### Читаємо таблицю

Найпоширеніше формулювання — «українська дорожча в токенах» — правдиве, але
марне: воно не каже, **наскільки** і **від чого це залежить**.

А залежить від однієї речі: **скільки місця словник виділив цій мові**. При
пʼятистах рядках українська дорожча майже на третину. При шістнадцяти тисячах —
на десяту частину. Націнка впала майже втричі, і мова при цьому не змінилась
жодною літерою.

Звідси й практичний висновок: коли модель погано працює з українською, питання
не «чи вміє вона українську», а **скільки рядків словника їй дісталося**.

## 7 · З чого складається націнка: довжина слів чи морфологія?

Націнку легко пояснити морфологією — мовляв, багаті закінчення змушують різати
дрібніше. Але є й простіше пояснення: **українське слово просто довше**. Ми вже
бачили відношення довжин на початку зошита.

Розділити ці два внески можна точно, і для цього досить арифметики. Ціну тисячі
слів запишемо як добуток двох множників:

```
токенів     токенів     символів
────────  = ────────  ×  ────────
 слово      символ        слово
```

Тоді націнка теж розкладається в **добуток двох націнок**:

- **націнка за довжину** — у скільки разів українське слово довше в символах.
  Вона від словника не залежить узагалі;
- **націнка за дрібність** — у скільки разів дрібніше токенізатор ріже **той самий
  символ**. Ось вона й відповідає за морфологію.

In [ ]:
length_factors = []
for seed in SEEDS:
    _, _, english_test, ukrainian_test = split_corpus(seed)
    english_length = sum(len(t) for t in english_test) / sum(len(t.split()) for t in english_test)
    ukrainian_length = sum(len(t) for t in ukrainian_test) / sum(len(t.split()) for t in ukrainian_test)
    length_factors.append(ukrainian_length / english_length)

length_factor, length_spread = mean_and_spread(length_factors)
print("націнка за довжину слова: x%.4f ± %.4f (від словника не залежить)" % (length_factor, length_spread))
print()

print("%7s %12s %12s %12s %12s" % ("словник", "за довжину", "за дрібність", "разом", "перевірка"))
grain_table = {}
for vocab_size in VOCAB_SIZES:
    rows = sweep[vocab_size]
    grains = [r[3] / r[2] for r in rows]           # токенів на символ: укр / англ
    totals = [r[1] / r[0] for r in rows]           # токенів на слово:  укр / англ
    grain_mean, grain_spread = mean_and_spread(grains)
    total_mean, _ = mean_and_spread(totals)
    grain_table[vocab_size] = (grain_mean, grain_spread)
    print("%7d %11.4f %12.4f %12.4f %12.4f" % (
        vocab_size, length_factor, grain_mean, total_mean, length_factor * grain_mean))

# розклад мусить бути точним для кожного зерна окремо, а не лише в середньому
for seed_index, seed in enumerate(SEEDS):
    row = sweep[16000][seed_index]
    predicted = length_factors[seed_index] * (row[3] / row[2])
    actual = row[1] / row[0]
    assert abs(predicted - actual) < 1e-9, "розклад націнки на два множники не зійшовся!"
print()
print("✅ добуток двох множників точно дорівнює повній націнці на кожному зерні")

### Ось це найцікавіше число теми

Дивись на колонку «за дрібність». При маленькому словнику вона більша за одиницю:
українську справді ріжуть дрібніше — на кожен символ виходить більше токенів.

Але зі зростанням словника вона **падає нижче одиниці**. При словнику на шістнадцять
тисяч українська ріжеться **грубше** за англійську на той самий символ.

Тобто вся націнка, яка лишається при великому словнику, — це **не морфологія, а
просто довші слова**. Морфологічна складова не те що зникає: вона міняє знак.

Пояснення просте, коли його побачив. Українських словоформ більше, але вони
**передбачувані**: ті самі десять закінчень чіпляються до тисяч основ. Щойно
словнику вистачає місця, щоб узяти ці закінчення окремими рядками, довгі шматки
української стають дешевими.

## 8 · Токенізатор знає домен, а не мову

Досі ми говорили «токенізатор для української». Це неточно, і зараз буде видно
чому. Візьмімо BPE зі словником на чотири тисячі, навчений на нашому корпусі, і
дамо йому кілька звичайних українських слів.

In [ ]:
DOMAIN_WORDS = ["телефон", "телефони", "телефонів", "налаштування",
                "неможливо", "розпакування", "файл", "дерево", "кіт"]

domain_report = collections.defaultdict(list)
for seed in SEEDS:
    domain_tokenizer = get_tokenizer("укр", 4000, seed)
    for word in DOMAIN_WORDS:
        pieces = domain_tokenizer.encode(word).tokens
        domain_report[word].append(pieces)

print("%-14s %-30s %8s %12s" % ("слово", "як поділено", "шматків", "у корпусі"))
for word in DOMAIN_WORDS:
    variants = domain_report[word]
    pieces = variants[0]
    stable = all(v == pieces for v in variants)
    print("%-14s %-30s %8d %12d%s" % (
        word, " | ".join(pieces), len(pieces), corpus_word_counts[word],
        "" if stable else "  (зерна розійшлися)"))

### Головний абзац теми

**«Телефон» розпадається на чотири шматки, а «налаштування» лишається цілим.**

Не тому, що одне слово складніше за інше — обидва звичайні українські іменники,
обидва довгі. А тому, що наш корпус складається з перекладів компʼютерних
інтерфейсів. У ньому «налаштування» трапляється сотні разів, а «телефон» —
одиниці. «Кіт» — слово, яке знає кожна дитина, — не трапляється **жодного разу**
і теж розсипається.

Це та сама думка, до якої курс компʼютерного зору йшов цілим блоком:
**модель знає той домен, на якому її навчили**. Токенізатор — найдешевша її
демонстрація, бо тут усе видно на одному слові, без жодного навчання мережі.

І практичний наслідок: **словник токенізатора — це відбиток корпусу**. Дивлячись
на нього, можна здогадатися, чим займалась модель, ще до першого запуску.

In [ ]:
# скільки взагалі слів лишаються цілими, а скільки розсипаються на шматки
whole_word_share = {}
for language, column in (("англ", 0), ("укр", 1)):
    shares = collections.defaultdict(list)
    for seed in SEEDS:
        parts = split_corpus(seed)
        test_texts = parts[2] if column == 0 else parts[3]
        tokenizer = get_tokenizer(language, 4000, seed)

        # рахуємо кожне слово стільки разів, скільки воно трапилось у тесті
        test_word_counts = collections.Counter()
        for text in test_texts:
            test_word_counts.update(word for word, _ in WORD_SPLIT.pre_tokenize_str(text))
        unique_words = list(test_word_counts)
        encoded_words = tokenizer.encode_batch(unique_words)

        histogram = collections.Counter()
        for word, encoded in zip(unique_words, encoded_words):
            histogram[min(len(encoded.ids), 4)] += test_word_counts[word]
        total_words = sum(histogram.values())
        for pieces in (1, 2, 3, 4):
            shares[pieces].append(histogram[pieces] / total_words)

    whole_word_share[language] = mean_and_spread(shares[1])[0]
    print("%s (слововживань у тесті: %d)" % (language, total_words))
    for pieces in (1, 2, 3, 4):
        label = "%d шматок " % pieces if pieces == 1 else ("%d шматки" % pieces if pieces < 4 else "4 і більше")
        share_mean, share_spread = mean_and_spread(shares[pieces])
        print("   %-12s %.4f ± %.4f" % (label, share_mean, share_spread))

**Англійське слово лишається цілим у 87 випадках зі ста, українське — у 74.**
І розсипається на чотири шматки й більше українське слово втричі частіше. Це та
сама націнка, тільки видно, звідки саме вона береться.

## 9 · WordPiece проти BPE

BPE — не єдиний спосіб. `BERT` побудований на **WordPiece**, і різниця між ними
здається принциповою: BPE зливає **найчастішу** пару, WordPiece — ту, що найбільше
збільшує правдоподібність корпусу, тобто пару з найкращим відношенням
`частота пари / (частота лівого × частота правого)`.

Звучить як зовсім інший алгоритм. Заміряймо, чи це видно в числах.

In [ ]:
compare = {"англ": [], "укр": []}
for seed in SEEDS:
    english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(seed)
    for language, train_texts, test_texts in (
        ("англ", english_train, english_test),
        ("укр", ukrainian_train, ukrainian_test),
    ):
        bpe_cost = tokens_per_1000_words(get_tokenizer(language, 4000, seed), test_texts)
        wordpiece_cost = tokens_per_1000_words(train_wordpiece(train_texts, 4000), test_texts)
        compare[language].append((bpe_cost, wordpiece_cost))

print("%6s %16s %16s %12s" % ("мова", "BPE / 1000 слів", "WordPiece", "різниця"))
for language in ("англ", "укр"):
    rows = compare[language]
    bpe_mean, bpe_spread = mean_and_spread([r[0] for r in rows])
    wp_mean, wp_spread = mean_and_spread([r[1] for r in rows])
    difference = wp_mean - bpe_mean
    print("%6s %9.1f ± %-4.2f %9.1f ± %-4.2f %8.1f  (%+.1f %%)" % (
        language, bpe_mean, bpe_spread, wp_mean, wp_spread,
        difference, 100.0 * (wp_mean / bpe_mean - 1)))
    print("       розкид по зернах %.2f, різниця %.1f — %s" % (
        max(bpe_spread, wp_spread), abs(difference),
        "різниця справжня" if abs(difference) > 3 * max(bpe_spread, wp_spread) else "у межах розкиду"))

In [ ]:
_, ukrainian_train, _, _ = split_corpus(0)
bpe_tokenizer = get_tokenizer("укр", 4000, 0)
wordpiece_tokenizer = train_wordpiece(ukrainian_train, 4000)

print("%-14s %-26s %s" % ("слово", "BPE", "WordPiece"))
for word in ["телефон", "телефонів", "розпакування", "налаштування", "кіт"]:
    print("%-14s %-26s %s" % (
        word,
        " | ".join(bpe_tokenizer.encode(word).tokens),
        " | ".join(wordpiece_tokenizer.encode(word).tokens)))

bpe_vocabulary = set(bpe_tokenizer.get_vocab())
wordpiece_vocabulary = set(token.replace("##", "") for token in wordpiece_tokenizer.get_vocab())
print()
print("спільних рядків у двох словниках (без позначки ##): %d із 4000" % len(bpe_vocabulary & wordpiece_vocabulary))

### Висновок, який варто сказати прямо

Різниця **є**, вона стабільна на всіх зернах і в кілька разів більша за розкид —
але вона маленька: близько чотирьох-пʼяти відсотків не на користь WordPiece.
Три чверті словника в них узагалі однакові.

Тобто вибір між BPE й WordPiece — це не вибір між добрим і поганим методом. Це
питання сумісності з готовою моделлю: береш `BERT` — береш WordPiece, бо словник
уже такий. Виграти на цьому виборі майже нічого.

Помітна тут інша дрібниця: WordPiece позначає **продовження слова** двома
ґратками (`##ування`). У BPE такої позначки немає, і межу слова доводиться
відновлювати інакше.

## 10 · Спецтокени: рядки, яких у тексті немає

У словнику моделі живуть не лише шматки слів. Там є кілька службових рядків, які
**ніколи не зустрічаються в тексті** й потрапляють у послідовність тільки тому, що
їх туди поставили навмисно.

| токен | навіщо |
|---|---|
| `[UNK]` | заміна символу, якого модель не знає взагалі |
| `[CLS]` | порожнє місце на початку, куди модель складе підсумок усього тексту |
| `[SEP]` | межа: кінець тексту або шов між двома текстами в парі |
| `[PAD]` | набивка, щоб усі рядки в пачці мали однакову довжину |
| `[MASK]` | дірка, яку модель має вгадати під час навчання |

Порядок важливий: спецтокени додають **першими**, тому вони дістають найменші
номери. Так номер `[PAD]` не зʼїде, якщо словник колись переучити.

In [ ]:
SPECIAL_TOKENS = ["[CLS]", "[SEP]", "[PAD]", "[MASK]"]

_, ukrainian_train, _, _ = split_corpus(0)
special_tokenizer = train_bpe(ukrainian_train, 4000, extra_specials=SPECIAL_TOKENS)

print("розмір словника:", special_tokenizer.get_vocab_size())
for token in ["[UNK]"] + SPECIAL_TOKENS:
    print("   %-8s номер %d" % (token, special_tokenizer.token_to_id(token)))

# розмітка: [CLS] на початку, [SEP] на межах
special_tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", special_tokenizer.token_to_id("[CLS]")),
                    ("[SEP]", special_tokenizer.token_to_id("[SEP]"))],
)

single = special_tokenizer.encode("Не вдалося відкрити файл")
print()
print("одне речення:", single.tokens)

pair = special_tokenizer.encode("Не вдалося відкрити файл", "Перевір права доступу")
print("пара речень: ", pair.tokens)
print("до якого з двох належить токен:", pair.type_ids)

In [ ]:
# набивка: коротший рядок доганяє довший, а маска каже, де справжні токени
special_tokenizer.enable_padding(
    pad_id=special_tokenizer.token_to_id("[PAD]"), pad_token="[PAD]", length=12)

for encoded in special_tokenizer.encode_batch(["файл", "Не вдалося відкрити файл"]):
    print(encoded.tokens)
    print("   маска уваги:", encoded.attention_mask)

special_tokenizer.no_padding()
print()
print("маска потрібна саме для того, щоб модель не рахувала набивку за текст")

### Пастка, про яку варто знати одразу

In [ ]:
print(special_tokenizer.encode("текст [CLS] далі").tokens)

Спецтокен, **написаний літерами прямо в тексті**, перетворюється на справжній
спецтокен. Для моделі це той самий рядок, що й службовий, — вона не має способу
відрізнити.

Звідси проста гігієна: текст, який приходить ззовні, перед токенізацією чистять
від рядків на кшталт `[CLS]`. Інакше зловмисник напише їх у своєму повідомленні й
змінить розмітку пари. У [темі 26](../26-bert/lecture.html), де ця розмітка
почне щось означати, ціна такої дірки стане зрозумілою.

## 11 · Один словник на дві мови

Досі кожна мова мала **свій** токенізатор. Справжні моделі так не роблять: словник
один, і він мусить обслуговувати всі мови одразу. Заміряймо, чого це коштує.

In [ ]:
SHARED_SIZE = 8000
shared_rows = []
for seed in SEEDS:
    english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(seed)
    shared_tokenizer = train_bpe(english_train + ukrainian_train, SHARED_SIZE)
    shared_rows.append((
        tokens_per_1000_words(shared_tokenizer, english_test),
        tokens_per_1000_words(shared_tokenizer, ukrainian_test),
    ))

shared_english, shared_english_spread = mean_and_spread([r[0] for r in shared_rows])
shared_ukrainian, shared_ukrainian_spread = mean_and_spread([r[1] for r in shared_rows])
own_english, own_ukrainian = markup_table[SHARED_SIZE][0], markup_table[SHARED_SIZE][1]

print("%16s %14s %14s %10s" % ("словник 8000", "англ", "укр", "націнка"))
print("%16s %14.1f %14.1f %9.1f %%" % ("свій на мову", own_english, own_ukrainian,
                                       100.0 * (own_ukrainian / own_english - 1)))
print("%16s %14.1f %14.1f %9.1f %%" % ("спільний", shared_english, shared_ukrainian,
                                       100.0 * (shared_ukrainian / shared_english - 1)))
print()
print("розкид по зернах: англ %.2f, укр %.2f" % (shared_english_spread, shared_ukrainian_spread))
print("англійська подорожчала на %.1f %%" % (100.0 * (shared_english / own_english - 1)))
print("українська подорожчала на %.1f %%" % (100.0 * (shared_ukrainian / own_ukrainian - 1)))

Платять **обидві** мови, і англійська навіть більше. Це прямий наслідок того,
що місце в словнику скінченне: дві мови ділять одні й ті самі вісім тисяч рядків.

Зате націнка на українську при спільному словнику виходить **меншою**, ніж при
двох окремих того самого розміру. Причина та сама, що в розділі 7: спільний
словник забирає в англійської довгі рідкісні шматки й віддає місце частим
українським закінченням.

## 12 · Ціна довгого тексту

Досі націнка була академічною цифрою. Тепер зробимо її грошима й обмеженнями.

Довжина в токенах вирішує дві дуже практичні речі:

- **скільки коштує запит** — платні сервіси рахують саме токени;
- **скільки тексту взагалі влізе** — вікно контексту моделі міряють у токенах,
  а не в словах.

Найгірший реальний випадок: токенізатор навчений **лише на англійській**, а текст
йому дають український. Спершу спробуймо це буквально.

In [ ]:
english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(0)
english_only = train_bpe(english_train, 8000)

unknown_id = english_only.token_to_id("[UNK]")
encoded = english_only.encode_batch(ukrainian_test)
total_tokens = sum(len(item.ids) for item in encoded)
unknown_tokens = sum(item.ids.count(unknown_id) for item in encoded)

print("англійський словник на українському тексті:")
print("   токенів на 1000 слів: %.1f" % tokens_per_1000_words(english_only, ukrainian_test))
print("   з них [UNK]:          %.4f" % (unknown_tokens / total_tokens))
print()
print("приклад:", english_only.encode("Не вдалося відкрити файл").tokens)

### Це не «дорого», це «сліпо»

Дев'ять токенів із десяти — `[UNK]`. Такий токенізатор не бачить українського
тексту взагалі: для моделі це рядок з однакових невідомих плям, у якому не
відрізнити «файл» від «дерево».

Саме тому справжні моделі так не роблять. Вони працюють не з символами, а з
**байтами**: будь-який символ Unicode розкладається на байти, а байтів усього 256,
і всі вони в словнику є **завжди**. Невідомого не буває за побудовою — але
кириличний символ коштує **два байти замість одного**.

Заміряймо цей, реалістичний, варіант.

In [ ]:
def train_byte_level_bpe(texts, vocab_size):
    """BPE поверх байтів — так побудовані GPT-подібні моделі. [UNK] тут не буває."""
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=False,
    )
    tokenizer.train_from_iterator(texts, trainer)
    return tokenizer


byte_rows = []
for seed in SEEDS:
    english_train, ukrainian_train, english_test, ukrainian_test = split_corpus(seed)
    byte_english = train_byte_level_bpe(english_train, 8000)
    byte_ukrainian = train_byte_level_bpe(ukrainian_train, 8000)
    byte_rows.append((
        tokens_per_1000_words(byte_english, english_test),      # англ своїм
        tokens_per_1000_words(byte_ukrainian, ukrainian_test),  # укр своїм
        tokens_per_1000_words(byte_english, ukrainian_test),    # укр англійським
    ))
    print("зерно", seed, "готове, минуло %.0f с" % (time.time() - STARTED_AT))

byte_english_cost, byte_english_spread = mean_and_spread([r[0] for r in byte_rows])
byte_ukrainian_cost, byte_ukrainian_spread = mean_and_spread([r[1] for r in byte_rows])
byte_foreign_cost, byte_foreign_spread = mean_and_spread([r[2] for r in byte_rows])

print()
print("байтовий BPE, словник 8000, токенів на 1000 слів:")
print("   англійський текст англійським словником: %8.1f ± %.2f" % (byte_english_cost, byte_english_spread))
print("   український текст українським словником: %8.1f ± %.2f" % (byte_ukrainian_cost, byte_ukrainian_spread))
print("   український текст АНГЛІЙСЬКИМ словником: %8.1f ± %.2f" % (byte_foreign_cost, byte_foreign_spread))
print()
print("множник до свого словника:        x%.3f" % (byte_foreign_cost / byte_ukrainian_cost))
print("множник до англійського тексту:   x%.3f" % (byte_foreign_cost / byte_english_cost))

### Переведемо в те, що видно користувачеві

Візьмімо вікно контексту на 4096 токенів — скромне за сьогоднішніми мірками.
Скільки слів у нього влізе?

In [ ]:
CONTEXT_WINDOW = 4096

print("у вікно на %d токенів влізе слів:" % CONTEXT_WINDOW)
for label, cost in (
    ("англійський текст, свій словник", byte_english_cost),
    ("український текст, свій словник", byte_ukrainian_cost),
    ("український текст, англійський словник", byte_foreign_cost),
):
    print("   %-40s %6.0f" % (label, CONTEXT_WINDOW / cost * 1000))

print()
print("той самий текст українською коштує в %.2f раза дорожче,"
      % (byte_foreign_cost / byte_ukrainian_cost))
print("якщо словник навчали не на ній")

Ось чим насправді є «націнка на українську». Не десять відсотків із таблиці
розділу 6 — та таблиця чесно давала кожній мові **свій** словник. А в кілька разів,
якщо словник дістався іншій мові.

І це вимірний аргумент, а не скарга: коли модель погано працює з українською,
першим ділом варто подивитись не на архітектуру, а на те, **скільки рядків словника
дісталося кирилиці**.

## 13 · Що з цього забрати

Заміряно в цьому зошиті:

1. **Субслова виграють не швидкістю, а дірками.** Словник зі слів для української
   в півтора раза більший за англійський на тому самому змісті — і при цьому
   дірявіший. При однаковому бюджеті в 4000 рядків слова втрачають кожне шосте
   українське слововживання, субслова — жодного.
2. **BPE — це тридцять рядків коду.** Наш власний ділить слова точно так само, як
   бібліотечний, хоча порядок злиттів у нього інший: на нічиїх кожна реалізація
   обирає по-своєму.
3. **Націнка на українську падає зі словником** майже втричі — з третини до
   десятої частини.
4. **І та, що лишається, — не морфологія, а довжина слів.** Множник «за дрібність»
   при великому словнику падає нижче одиниці: українську ріжуть **грубше** за
   англійську на той самий символ.
5. **Токенізатор знає домен, а не мову.** «Налаштування» ціле, «телефон» на чотири
   шматки, «кіт» на два — бо корпус складається з перекладів інтерфейсів.
6. **WordPiece програє BPE близько пʼяти відсотків** — стабільно, але мало. Три
   чверті словника в них однакові.
7. **Чужий словник — це не «дорого», а «сліпо»** (дев'ять токенів із десяти `[UNK]`),
   а байтовий — у кілька разів дорожче.

In [ ]:
print("зошит виконано за %.0f с" % (time.time() - STARTED_AT))

## Завдання

### 🟢 Рівень 1
Додай до розділу 6 сьомий розмір словника — 32000 — і встав його в таблицю.
Перевір, чи націнка продовжує падати, чи вийшла на полицю.

### 🟡 Рівень 2
Візьми з корпусу дві програми з різних доменів (наприклад, `gimp` і `gcc`), навчи
на кожній свій BPE зі словником 2000 і подивись, як кожен із них ділить слова
іншого домену. Знайди слово, яке в одного ціле, а в іншого розсипається.

### 🔴 Рівень 3
Наш BPE у розділі 5 працює зі словником частот. Допиши до нього **кодувальник**:
функцію, яка бере нове слово, якого в навчальному словнику не було, і застосовує
до нього вивчені злиття **в тому порядку, у якому їх вивчено**. Звір результат із
`tokenizer.encode()` на сотні випадкових слів корпусу.

Повний текст завдань — у [homework.html](homework.html).